In [1]:
# 安裝必要的套件
!pip install tensorflow opencv-python gradio openai

# 匯入必要的套件
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import random
from tensorflow import keras
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from google.colab import drive
import os
import zipfile
import shutil
import gradio as gr
import openai

In [2]:
# 設定 OpenAI 的 API 金鑰（從 Colab 左側欄的 Secrets 讀取，名稱為 OPENAI_API_KEY）
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get('OPENAI_API_KEY')
)

In [3]:
category_en = "concrete,crack,efflorescence"
categories = category_en.split(',')
category_zh = "鏽蝕,裂縫,壁癌"
labels = category_zh.split(',')
num_classes = len(categories)

In [4]:
# 創建類別目錄
base_dir = '/content/'
for category in categories:
    os.makedirs(os.path.join(base_dir, category, 'images'), exist_ok=True)
    os.makedirs(os.path.join(base_dir, category, 'masks'), exist_ok=True)

In [5]:
!wget --no-check-certificate \
    "https://github.com/Yunyuhu/dtaidataset/raw/main/dataset.zip" \
    -O /content/dataset.zip

local_zip = '/content/dataset.zip'
with zipfile.ZipFile(local_zip, 'r') as zip_ref:
    zip_ref.extractall('/content')

In [6]:
# 移動解壓縮的檔案到相應的資料夾
for category in categories:
    source_dir = os.path.join('/content', category)
    target_images_dir = os.path.join(base_dir, category, 'images')
    if os.path.exists(source_dir):
        for file_name in os.listdir(source_dir):
            file_path = os.path.join(source_dir, file_name)
            if os.path.isfile(file_path):
                shutil.move(file_path, target_images_dir)

In [7]:
# 將正常圖片轉換為二進制圖片
def convert_to_binary(image_path, output_path, threshold=127):
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error loading image: {image_path}")
        return

    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, binary_image = cv2.threshold(gray_image, threshold, 255, cv2.THRESH_BINARY)
    cv2.imwrite(output_path, binary_image)

def process_images_and_create_masks(image_dir, mask_dir, image_size=224):
    if not os.path.exists(mask_dir):
        os.makedirs(mask_dir)

    image_files = os.listdir(image_dir)
    for image_file in image_files:
        image_path = os.path.join(image_dir, image_file)
        mask_path = os.path.join(mask_dir, image_file)
        convert_to_binary(image_path, mask_path)

# 處理每個資料夾中的圖片並創建對應的 mask
for category in categories:
    images_path = os.path.join(base_dir, category, "images")
    masks_path = os.path.join(base_dir, category, "masks")
    process_images_and_create_masks(images_path, masks_path)

# 收集所有類別的圖片和 mask 路徑
all_image_ids = []
for category in categories:
    images_path = os.path.join(base_dir, category, "images")
    image_files = [os.path.splitext(f)[0] for f in os.listdir(images_path)]
    all_image_ids.extend([(category, f) for f in image_files])

In [8]:
# 資料前處理函數
def load_image(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (224, 224))
    img = img / 255.0
    img = np.expand_dims(img, axis=-1)
    return img

def load_mask(mask_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (224, 224))
    # mask = mask / 255.0
    # mask = np.expand_dims(mask, axis=-1)
    mask[mask >= num_classes] = num_classes - 1
    mask = keras.utils.to_categorical(mask, num_classes=num_classes)
    return mask

In [9]:
# 資料增強
def augment_data(image, mask):
    data_gen_args = dict(horizontal_flip=True, vertical_flip=True, rotation_range=90)
    image_datagen = ImageDataGenerator(**data_gen_args)
    mask_datagen = ImageDataGenerator(**data_gen_args)
    seed = random.randint(1, 1000)
    image = next(image_datagen.flow(np.expand_dims(image, axis=0), batch_size=1, seed=seed))[0]
    mask = next(mask_datagen.flow(np.expand_dims(mask, axis=0), batch_size=1, seed=seed))[0]
    return image, mask

In [10]:
def create_dataset(image_ids, batch_size=4, num_classes=3, augment=False):
    def generator():
      while True:
          np.random.shuffle(image_ids)
          for start in range(0, len(image_ids), batch_size):
              x_batch = []
              y_batch = []
              end = min(start + batch_size, len(image_ids))
              batch_image_ids = image_ids[start:end]
              for (dataset_name, image_id) in batch_image_ids:
                  img_path = os.path.join("/content", dataset_name, "images", image_id + ".jpg")
                  mask_path = os.path.join("/content", dataset_name, "masks", image_id + ".jpg")
                  image = load_image(img_path)
                  mask = load_mask(mask_path)
                  if augment:
                      image, mask = augment_data(image, mask)
                  x_batch.append(image)
                  y_batch.append(mask)
              if x_batch and y_batch:  # 確保批次不為空
                    yield np.array(x_batch), np.array(y_batch)
    return generator()

In [11]:
# 拆分訓練集和驗證集
train_size = int(0.8 * len(all_image_ids))
train_ids = all_image_ids[:train_size]
val_ids = all_image_ids[train_size:]

steps_per_epoch = len(train_ids) // 4  # 確保步數與批次大小一致
validation_steps = len(val_ids) // 4  # 確保步數與批次大小一致

train_gen = create_dataset(train_ids, batch_size=4, num_classes=num_classes, augment=True)
val_gen = create_dataset(val_ids, batch_size=4, num_classes=num_classes)

In [12]:
# 定義 U-Net 模型
def unet(input_size=(224, 224, 1), num_classes=3):
    inputs = Input(input_size)

    conv1 = Conv2D(64, 3, activation='relu', padding='same')(inputs)
    conv1 = Conv2D(64, 3, activation='relu', padding='same')(conv1)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = Conv2D(128, 3, activation='relu', padding='same')(pool1)
    conv2 = Conv2D(128, 3, activation='relu', padding='same')(conv2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = Conv2D(256, 3, activation='relu', padding='same')(pool2)
    conv3 = Conv2D(256, 3, activation='relu', padding='same')(conv3)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)

    conv4 = Conv2D(512, 3, activation='relu', padding='same')(pool3)
    conv4 = Conv2D(512, 3, activation='relu', padding='same')(conv4)
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)

    conv5 = Conv2D(1024, 3, activation='relu', padding='same')(pool4)
    conv5 = Conv2D(1024, 3, activation='relu', padding='same')(conv5)

    up6 = UpSampling2D(size=(2, 2))(conv5)
    merge6 = concatenate([conv4, up6], axis=3)
    conv6 = Conv2D(512, 3, activation='relu', padding='same')(merge6)
    conv6 = Conv2D(512, 3, activation='relu', padding='same')(conv6)

    up7 = UpSampling2D(size=(2, 2))(conv6)
    merge7 = concatenate([conv3, up7], axis=3)
    conv7 = Conv2D(256, 3, activation='relu', padding='same')(merge7)
    conv7 = Conv2D(256, 3, activation='relu', padding='same')(conv7)

    up8 = UpSampling2D(size=(2, 2))(conv7)
    merge8 = concatenate([conv2, up8], axis=3)
    conv8 = Conv2D(128, 3, activation='relu', padding='same')(merge8)
    conv8 = Conv2D(128, 3, activation='relu', padding='same')(conv8)

    up9 = UpSampling2D(size=(2, 2))(conv8)
    merge9 = concatenate([conv1, up9], axis=3)
    conv9 = Conv2D(64, 3, activation='relu', padding='same')(merge9)
    conv9 = Conv2D(64, 3, activation='relu', padding='same')(conv9)

    conv10 = Conv2D(num_classes, 1, activation='softmax')(conv9)

    model = Model(inputs, conv10)
    model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [13]:
# 建立模型
model = unet(num_classes=3)
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 1)]        0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 224, 224, 64)         640       ['input_1[0][0]']             
                                                                                                  
 conv2d_1 (Conv2D)           (None, 224, 224, 64)         36928     ['conv2d[0][0]']              
                                                                                                  
 max_pooling2d (MaxPooling2  (None, 112, 112, 64)         0         ['conv2d_1[0][0]']            
 D)                                                                                           

In [14]:
# 訓練模型
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=10, verbose=1),
    ModelCheckpoint('/content/unet_model.h5', verbose=1, save_best_only=True, save_weights_only=False),
    ReduceLROnPlateau(factor=0.1, patience=3, min_lr=1e-7, verbose=1)
]

steps_per_epoch = len(train_ids)
validation_steps = len(val_ids)

history = model.fit(train_gen, steps_per_epoch=steps_per_epoch, epochs=5, validation_data=val_gen, validation_steps=validation_steps, callbacks=callbacks)

Epoch 1/5
2400/2400 [==============================] - ETA: 0s - loss: 0.2414 - accuracy: 0.8405
Epoch 1: val_loss improved from inf to 0.14790, saving model to /content/unet_model.h5


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


2400/2400 [==============================] - 638s 255ms/step - loss: 0.2414 - accuracy: 0.8405 - val_loss: 0.1479 - val_accuracy: 0.8884 - lr: 1.0000e-04
Epoch 2/5
2400/2400 [==============================] - ETA: 0s - loss: 0.1936 - accuracy: 0.8771
Epoch 2: val_loss improved from 0.14790 to 0.13454, saving model to /content/unet_model.h5
2400/2400 [==============================] - 578s 241ms/step - loss: 0.1936 - accuracy: 0.8771 - val_loss: 0.1345 - val_accuracy: 0.8962 - lr: 1.0000e-04
Epoch 3/5
2400/2400 [==============================] - ETA: 0s - loss: 0.1867 - accuracy: 0.8827
Epoch 3: val_loss did not improve from 0.13454
2400/2400 [==============================] - 568s 237ms/step - loss: 0.1867 - accuracy: 0.8827 - val_loss: 0.1355 - val_accuracy: 0.9144 - lr: 1.0000e-04
Epoch 4/5
2400/2400 [==============================] - ETA: 0s - loss: 0.1832 - accuracy: 0.8857
Epoch 4: val_loss improved from 0.13454 to 0.12511, saving model to /content/unet_model.h5
2400/2400 [=======

In [15]:
# 保存模型
model.save('/content/unet_model.h5')

# 加載模型
model = tf.keras.models.load_model('/content/unet_model.h5')

# 評估模型
loss, accuracy = model.evaluate(val_gen, steps=validation_steps)
print(f"Validation Loss: {loss*100:.2f}%")
print(f"Validation Accuracy: {accuracy*100:.2f}%")

600/600 [==============================] - 45s 74ms/step - loss: 0.1240 - accuracy: 0.9042
Validation Loss: 12.40%
Validation Accuracy: 90.42%


In [16]:
# 推論測試（取驗證集的第一張圖）
category, image_id = val_ids[0]
test_image_path = os.path.join(base_dir, category, "images", image_id + ".jpg")
test_image = load_image(test_image_path)
pred_mask = model.predict(np.expand_dims(test_image, axis=0))

1/1 [==============================] - 2s 2s/step


In [25]:
 # 定義獲得GPT建議的函數
def classify_image_and_get_advice(main_feature_str):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        temperature=0.5,
        max_tokens=120,
        messages=[
            {"role": "system", "content": "For the wall of the house condition problem, please give the corresponding characteristics of the home treatment and list of recommendations. Please be short."},
            {"role": "user", "content": main_feature_str}
        ]
    )

    return response.choices[0].message.content

In [26]:
# 定義處理上傳圖片的函數
def predict_and_display(image):
    test_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    test_image = cv2.resize(test_image, (224, 224))
    test_image = test_image / 255.0
    test_image = np.expand_dims(test_image, axis=-1)

    pred_mask = model.predict(np.expand_dims(test_image, axis=0))
    pred_mask = np.argmax(pred_mask, axis=-1).astype(np.uint8)

    pixel_width_mm = 0.5
    pixel_height_mm = 0.5

    areas = {}
    dimensions = {}

    for i, label in enumerate(labels):
        mask = (pred_mask == i).astype(np.uint8)
        area_pixels = np.sum(mask)
        area_mm2 = area_pixels * pixel_width_mm * pixel_height_mm

        if mask.ndim == 2:
            height_pixels = np.sum(np.any(mask, axis=1))
            width_pixels = np.sum(np.any(mask, axis=0))
            height_mm = height_pixels * pixel_height_mm
            width_mm = width_pixels * pixel_width_mm
        else:
            height_mm = 0
            width_mm = 0

        areas[label] = area_mm2
        dimensions[label] = (width_mm, height_mm)

    main_feature_str = ", ".join(f"{label}: {area:.2f} mm²" for label, area in areas.items())
    advice = classify_image_and_get_advice(main_feature_str)

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 4, 1)
    plt.title('Original Image')
    plt.imshow(test_image.squeeze(), cmap='gray')

    for i, category in enumerate(categories):
        plt.subplot(1, 4, i + 2)
        plt.title(f'{category} Mask')
        plt.imshow((pred_mask.squeeze() == i).astype(np.uint8), cmap='gray')

    plt.tight_layout()
    plt.show()

    areas_str = "\n".join([f"{label}: {area:.2f} mm²" for label, area in areas.items()])
    return plt, areas_str, advice

In [27]:
# 使用 Gradio 建立界面
interface = gr.Interface(
    fn=predict_and_display,
    inputs=gr.Image(type="numpy", label="Upload an image"),
    outputs=[
        gr.Plot(label="Predicted Mask"),
        gr.Textbox(label="Detected Areas (in square millimeters)"),
        gr.Textbox(label="AI Recommendations")
    ],
    title="牆牆出事 Wall Damage Detection",
    description="Upload an image of a wall to detect cracks, efflorescence, and corrosion areas."
)

# 啟動界面
interface.launch(debug=True)